In [ ]:

import sys
from pathlib import Path

PROJECT_ROOT = Path("/workspace/ECG")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%matplotlib inline

print("PROJECT_ROOT:", PROJECT_ROOT)

from src.download_mitbih_paralel import download_mitbih_paralel
from src.download_incart_paralel import download_incart_paralel

from src.qc_dataset_ext import qc_dataset_ext
from src.plot_preprocessing_comparison import plot_preprocessing_comparison

from src.prepare_mitbih_datasets import prepare_mitbih_datasets
from src.prepare_incart_datasets import prepare_incart_datasets
from src.prepare_incart_datasets_12ch import prepare_incart_datasets_12ch
from src.prepare_cross_dataset_splits import prepare_cross_dataset_splits

from src.build_general_features import build_general_features
from src.build_morphology_features import build_morphology_features
from src.report_features import report_features

from src.train_featurext import train_featurext
from src.train_cnn import train_cnn

from src.plot_dataset_tsne import plot_dataset_tsne
from src.plot_cross_dataset_tsne import plot_cross_dataset_tsne
from src.plot_tsne_featurext_vs_cnn import plot_tsne_featurext_vs_cnn

from src.compare_models_extended import compare_dataset_models_extended

from src.notebook_utils import (
    run_step_if_needed,
    reset_section,
    checkpoint_path,
    load_checkpoint,
)

import src.config as cfg
from src.config import (
    OUTPUT_DIR,
    INTERIM_DIR,
    RAW_DIR,
    DEFAULT_FS,
    CNN_AUGMENTATION_CONFIG,
    PREPROCESSING_CONFIG,
    get_ds_par,
    get_label_mapping,
    get_label_mode,
    get_label_mode_classes,
)

print("RAW_DIR:", RAW_DIR)
print("INTERIM_DIR:", INTERIM_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("DEFAULT_FS:", DEFAULT_FS)
print("LABEL_MODE:", get_label_mode())
print("CLASS_NAMES:", get_label_mode_classes())
print("LABEL_MAPPING:", get_label_mapping())
print("PREPROCESSING_CONFIG:", PREPROCESSING_CONFIG)
print("CNN_AUGMENTATION_CONFIG:", CNN_AUGMENTATION_CONFIG)


# ------------------------------------------------------------------
# Opcionális runtime override
# ------------------------------------------------------------------
# Lehetséges értékek:
# - "aami5"
# - "binary_n_vs_rest"
# - "ternary_n_v_other"
#
# Példa:
# cfg.LABEL_MODE = "ternary_n_v_other"

# cfg.LABEL_MODE = "binary_n_vs_rest"

print("ACTIVE LABEL_MODE:", cfg.get_label_mode())
print("ACTIVE CLASS_NAMES:", cfg.get_label_mode_classes())

# ------------------------------------------------------------------
# Opcionális checkpoint reset
# ------------------------------------------------------------------
# Ha teljes újragenerálás kell az új label mode / resampling miatt:
#
# for sec in ["mitbih", "incart", "cross_test", "mixed", "domain_generalization", "compare"]:
#     reset_section(sec)


# ------------------------------------------------------------------
# MIT-BIH
# ------------------------------------------------------------------
SECTION = "mitbih"
COMMON = {"dataset": "mitbih"}

run_step_if_needed(SECTION, download_mitbih_paralel)
run_step_if_needed(SECTION, qc_dataset_ext, COMMON)
run_step_if_needed(SECTION, plot_preprocessing_comparison, {**COMMON, "record_name": "100"})

# Fontos: az új label mode + DEFAULT_FS miatt ezeket érdemes újrafuttatni
run_step_if_needed(SECTION, prepare_mitbih_datasets)
run_step_if_needed(SECTION, build_general_features, COMMON)
run_step_if_needed(SECTION, build_morphology_features, COMMON)
run_step_if_needed(SECTION, report_features, COMMON)

run_step_if_needed(SECTION, train_featurext, COMMON)
run_step_if_needed(SECTION, plot_dataset_tsne, COMMON)
run_step_if_needed(SECTION, plot_tsne_featurext_vs_cnn, COMMON)
run_step_if_needed(SECTION, train_cnn, COMMON)

# ------------------------------------------------------------------
# INCART
# ------------------------------------------------------------------
SECTION = "incart"
COMMON = {"dataset": "incart"}

run_step_if_needed(SECTION, download_incart_paralel)
run_step_if_needed(SECTION, qc_dataset_ext, COMMON)
run_step_if_needed(SECTION, plot_preprocessing_comparison, {**COMMON, "record_name": "I01"})

# 1 csatornás, resampled, új label mode
run_step_if_needed(SECTION, prepare_incart_datasets)
run_step_if_needed(SECTION, build_general_features, COMMON)
run_step_if_needed(SECTION, build_morphology_features, COMMON)
run_step_if_needed(SECTION, report_features, COMMON)

run_step_if_needed(SECTION, train_featurext, COMMON)
run_step_if_needed(SECTION, plot_dataset_tsne, COMMON)
run_step_if_needed(SECTION, plot_tsne_featurext_vs_cnn, COMMON)
run_step_if_needed(SECTION, train_cnn, {**COMMON, "variant": "1ch"}, step_name="train_cnn_1ch")

# 12 csatornás CNN külön
run_step_if_needed(SECTION, prepare_incart_datasets_12ch)
run_step_if_needed(SECTION, train_cnn, {**COMMON, "variant": "12ch"}, step_name="train_cnn_12ch")

# ------------------------------------------------------------------
# Cross-dataset split-ek
# ------------------------------------------------------------------
# Fontos:
# a MITBIH és INCART prepare lépéseknek ugyanazzal a LABEL_MODE-dal kell lefutniuk.
# prepare_cross_dataset_splits ezt feltételezi.

run_step_if_needed("cross_test", prepare_cross_dataset_splits)

SECTION = "cross_test"
COMMON = {"dataset": "cross_test"}
run_step_if_needed(SECTION, build_general_features, COMMON)
run_step_if_needed(SECTION, build_morphology_features, COMMON)
run_step_if_needed(SECTION, report_features, COMMON)
run_step_if_needed(SECTION, plot_dataset_tsne, COMMON)
run_step_if_needed(SECTION, train_featurext, COMMON)
run_step_if_needed(SECTION, plot_tsne_featurext_vs_cnn, COMMON)
run_step_if_needed(SECTION, train_cnn, COMMON)

SECTION = "mixed"
COMMON = {"dataset": "mixed"}
run_step_if_needed(SECTION, prepare_cross_dataset_splits)
run_step_if_needed(SECTION, build_general_features, COMMON)
run_step_if_needed(SECTION, build_morphology_features, COMMON)
run_step_if_needed(SECTION, report_features, COMMON)
run_step_if_needed(SECTION, plot_dataset_tsne, COMMON)
run_step_if_needed(SECTION, train_featurext, COMMON)
run_step_if_needed(SECTION, plot_tsne_featurext_vs_cnn, COMMON)
run_step_if_needed(SECTION, train_cnn, COMMON)

SECTION = "domain_generalization"
COMMON = {"dataset": "domain_generalization"}
run_step_if_needed(SECTION, prepare_cross_dataset_splits)
run_step_if_needed(SECTION, build_general_features, COMMON)
run_step_if_needed(SECTION, build_morphology_features, COMMON)
run_step_if_needed(SECTION, report_features, COMMON)
run_step_if_needed(SECTION, plot_dataset_tsne, COMMON)
run_step_if_needed(SECTION, train_featurext, COMMON)
run_step_if_needed(SECTION, plot_tsne_featurext_vs_cnn, COMMON)
run_step_if_needed(SECTION, train_cnn, COMMON)


SECTION = "compare"
# =========================================================
# 1. CROSS DATASET t-SNE (MITBIH vs INCART)
# =========================================================

dg_dir = Path(get_ds_par("domain_generalization", "beats_dir"))

run_step_if_needed(SECTION, plot_cross_dataset_tsne, {
    "source_dataset": "mitbih_train", "target_dataset": "incart_test",
    "source_path": dg_dir / "train.npz", "target_path": dg_dir / "test.npz",
    "max_per_dataset": 3000,
    }, step_name="tsne_domain_generalization_train_vs_test",)

# =========================================================
# 2. INCART MODEL ÖSSZEHASONLÍTÁSOK
# =========================================================

# featurext vs cnn
run_step_if_needed(SECTION, compare_dataset_models_extended, {
    "dataset": "incart",
    "left_family": "featurext", "right_family": "cnn",
    "select_best_left": True, "select_best_right": True,
    }, step_name="compare_incart_featurext_vs_cnn",)

# cnn vs cnn12
run_step_if_needed(SECTION, compare_dataset_models_extended, {
    "dataset": "incart",
    "left_family": "cnn", "right_family": "cnn12",
    "select_best_left": True, "select_best_right": True,
    }, step_name="compare_incart_cnn_vs_cnn12",)

# featurext vs cnn12
run_step_if_needed(SECTION, compare_dataset_models_extended, {
    "dataset": "incart",
    "left_family": "featurext", "right_family": "cnn12",
    "select_best_left": True, "select_best_right": True,
    }, step_name="compare_incart_featurext_vs_cnn12",)

# =========================================================
# 3. MITBIH ÖSSZEHASONLÍTÁS
# =========================================================

run_step_if_needed(SECTION, compare_dataset_models_extended, {
    "dataset": "mitbih",
    "left_family": "featurext", "right_family": "cnn",
    "select_best_left": True, "select_best_right": True,
    }, step_name="compare_mitbih_featurext_vs_cnn",)


